<a href="https://colab.research.google.com/github/kushal984762333-glitch/ibm-ds-capstone-spacex/blob/main/01-spacex-data-collection-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Data Collection with the SpaceX REST API
Requests launch data from `api.spacexdata.com/v4`, enriches it with rocket, launchpad, payload and core details, and saves a DataFrame.

In [1]:
import requests, pandas as pd, numpy as np
pd.set_option('display.max_columns',None)
spacex_url='https://api.spacexdata.com/v4/launches/past'
response=requests.get(spacex_url)
print(response.status_code)

522


import requests
import pandas as pd
import time

urls = [
    "https://api.spacexdata.com/v4/launches",
    "https://api.spacexdata.com/v4/launches/past"
]

data = None

for url in urls:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = pd.json_normalize(response.json())
        print("Data loaded successfully from:", url)
        break
    except (requests.RequestException, ValueError) as e:
        print(f"Failed to load from {url}: {e}")

if data is None or data.empty:
    print("Live API is unavailable. Loading static fallback dataset...")
    try:
        data = pd.read_json('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json')
        print("Static fallback dataset loaded successfully.")
    except Exception as e:
        print(f"Failed to load fallback dataset: {e}")

if data is not None:
    required_columns = [
        'rocket', 'payloads', 'launchpad',
        'cores', 'flight_number', 'date_utc'
    ]

    data = data[required_columns]

    data = data[data['cores'].map(len) == 1]
    data = data[data['payloads'].map(len) == 1]

    data['cores'] = data['cores'].map(lambda x: x[0])
    data['payloads'] = data['payloads'].map(lambda x: x[0])

    data['date'] = pd.to_datetime(data['date_utc']).dt.date

    data = data[
        data['date'] <= pd.Timestamp(2020, 11, 13).date()
    ]

    display(data.head())
else:
    print("Both API and fallback dataset are unavailable.")

In [7]:

import requests

try:
    r = requests.get("https://www.google.com", timeout=5)
    print("Internet works. Status:", r.status_code)
except Exception as e:
    print("Internet connection failed:", repr(e))

Internet works. Status: 200


In [8]:

import requests

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"

try:
    r = requests.get(url, timeout=10)
    print("Status:", r.status_code)
    print("Bytes received:", len(r.content))
    print("First 100 characters:", r.text[:100])
except Exception as e:
    print("Dataset download failed:", repr(e))


Status: 200
Bytes received: 269589
First 100 characters: [{"fairings": {"reused": false, "recovery_attempt": false, "recovered": false, "ships": []}, "links"
